# FT-Transformer V0 — ensemble diversity 평가

**목적**: v9(LightGBM×2 + CatBoost)와 **다른 feature representation** 을 가진 모델이
ensemble diversity 를 만들 수 있는지 평가합니다. v9 를 대체하려는 것이 아닙니다.

## 왜 이 질문인가

17차 MLP 변형 6종이 이 데이터에서 다음 직선 위에 놓였습니다.

```
BSS = 2052 × corr(v9) − 1051        R² = 0.998
```

정확도와 탈상관이 1:1 로 교환됩니다. 게이트는 `corr < 0.85` 에서 `BSS ≥ 750` 을
요구하는데 직선은 그 지점에서 **693.2** 를 예측하므로, 통과하려면
**직선에서 +57 BSS 이상 벗어나야** 합니다. attention 이 그 밖에 착지하는지만 봅니다.

## 이 노트북이 하지 않는 것

- **제출 파일에 영향 없음** — `work/submit_v9.zip` 을 열지도 수정하지도 않습니다
- **`test.csv` 미사용** — 학습 `fold_2024_tr`(시즌 2019–2023), 예측 `fold_2024_va`(2024)
- **`work/` 미사용** — v9 구성요소는 준비된 데이터 아카이브 안에 들어 있습니다
- **기존 pipeline 무수정** — `models.py` / `train.py` / `dataset.py` / `metrics.py` 등
- **제출 파일 생성 없음**

위에서 아래로 순서대로 실행하십시오. 예상 40~90분 (T4 기준).


## 1. Google Drive 연결


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Repository 준비

`/content/kbo` 가 있으면 `git pull --ff-only`, 없으면 clone 합니다.
**rebase / force 는 쓰지 않습니다.**


In [ ]:
import os, pathlib, subprocess

REPO_URL = 'https://github.com/Gromiit/kbo-control.git'
REPO = pathlib.Path('/content/kbo')

if (REPO / '.git').exists():
    print('기존 저장소 -> git pull --ff-only')
    print(subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'],
                         capture_output=True, text=True).stdout.strip())
else:
    print('clone')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
head = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                      capture_output=True, text=True).stdout.strip()
print(f'\ncwd    {os.getcwd()}')
print(f'commit {head}')

FTT = REPO / 'experiments' / 'deep_learning' / 'ft_transformer'
for f in ('train_ftt.py', 'v0_gate.py', 'requirements_colab.txt'):
    assert (FTT / f).exists(), f'없음: {FTT / f}'
    print(f'  OK  {f}')


## 3. GPU 확인

CUDA 가 없으면 **여기서 멈춥니다.** 이 실험은 MPS/CPU 학습을 하지 않습니다.


In [ ]:
import torch

ok = torch.cuda.is_available()
print(f'CUDA available : {ok}')
if ok:
    p = torch.cuda.get_device_properties(0)
    print(f'GPU            : {p.name}')
    print(f'CUDA version   : {torch.version.cuda}')
    print(f'VRAM           : {p.total_memory/1e9:.1f} GB')
    print(f'bf16           : {torch.cuda.is_bf16_supported()}')
else:
    raise RuntimeError('Colab GPU runtime required. MPS/CPU training is disabled.')


## 4. Dependency

Colab 은 네 패키지를 이미 갖고 있는 경우가 많습니다. 없을 때만 설치됩니다.


In [ ]:
!pip install -q -r experiments/deep_learning/ft_transformer/requirements_colab.txt

import importlib
for m in ('torch', 'numpy', 'pandas', 'pyarrow'):
    mod = importlib.import_module(m)
    print(f'  {m:10s} {getattr(mod, "__version__", "?")}')


## 5. Dataset 준비 및 계약 검증

Drive 의 `MyDrive/kbo/ftt_data.tgz` 를 풀어
`experiments/.../ft_transformer/data/` 에 놓습니다.

아카이브에는 **정확히 세 파일**만 들어 있어야 합니다.

```
data/ftt_2024_tr.parquet
data/ftt_2024_va.parquet
data/schema.json
```

> **크기로는 버전을 구분할 수 없습니다.** `schema.json` 이 없던 구버전
> 아카이브도 같은 260.3 MB 이고 금지항목 검사도 통과합니다. 그래서 이 셀은
> payload 를 **집합 일치**로 검사합니다. `schema.json` 이 없으면 **여기서
> 중단**하며, 추정해서 만들어 통과시키지 않습니다 — 학습 40~90분을 쓴 뒤
> 잘못된 데이터였음을 알게 되는 것보다 낫습니다.


In [ ]:
import hashlib, json, pathlib, tarfile

ARCHIVE = pathlib.Path('/content/drive/MyDrive/kbo/ftt_data.tgz')
FTT = pathlib.Path('/content/kbo/experiments/deep_learning/ft_transformer')
DATA = FTT / 'data'
assert DATA == pathlib.Path(
    '/content/kbo/experiments/deep_learning/ft_transformer/data'), DATA

CONTRACT = 'ftt-v0/1'
EXPECT = ['data/ftt_2024_tr.parquet', 'data/ftt_2024_va.parquet',
          'data/schema.json']
FORBIDDEN = ('test.csv', 'work/', '.pkl', '.cbm', '.zip', 'submit',
             'checkpoint', 'shard')

STALE = ('\n아카이브를 다시 만들어야 합니다. Mac 에서:\n'
         '    python experiments/deep_learning/ft_transformer/prep_data.py\n'
         '그 다음 Drive 의 MyDrive/kbo/ftt_data.tgz 를 덮어쓰고 '
         '이 노트북을 처음부터 실행하십시오.')

assert ARCHIVE.exists(), (
    f'아카이브가 없습니다: {ARCHIVE}\n'
    'Drive 를 마운트했는지, MyDrive/kbo/ 에 올렸는지 확인하십시오.')

# ---- 1. 아카이브 정체
h = hashlib.sha256()
with open(ARCHIVE, 'rb') as f:
    for blk in iter(lambda: f.read(1 << 22), b''):
        h.update(blk)
print(f'archive  {ARCHIVE}')
print(f'size     {ARCHIVE.stat().st_size:,} bytes '
      f'({ARCHIVE.stat().st_size/1e6:.1f} MB)')
print(f'sha256   {h.hexdigest()}')

# ---- 2. payload — 집합 일치. 구버전 아카이브는 여기서 걸린다
with tarfile.open(ARCHIVE) as t:
    names = sorted(m.name for m in t.getmembers() if m.isfile())
bad = [n for n in names if any(f in n for f in FORBIDDEN)]
assert not bad, f'금지 항목: {bad}'
if names != EXPECT:
    missing, extra = sorted(set(EXPECT) - set(names)), sorted(set(names) - set(EXPECT))
    raise AssertionError(
        f'payload 가 계약과 다릅니다 (contract {CONTRACT})\n'
        f'  기대  {EXPECT}\n  실제  {names}\n'
        + (f'  누락  {missing}\n' if missing else '')
        + (f'  초과  {extra}\n' if extra else '') + STALE)
print(f'payload  {names}')
print(f'         계약 일치 · 금지 항목 0개')

# ---- 3. schema.json 은 tar 안에서 직접 읽는다
with tarfile.open(ARCHIVE) as t:
    raw = t.extractfile('data/schema.json').read()
print(f'schema   sha256 {hashlib.sha256(raw).hexdigest()}')
sc = json.loads(raw)
need = ('contract', 'fold', 'numeric_features', 'n_numeric',
        'categorical_features', 'categorical_source', 'meta_columns', 'files')
miss = [k for k in need if k not in sc]
assert not miss, f'schema.json 필수 키 누락: {miss}{STALE}'
assert sc['contract'] == CONTRACT, (
    f'contract {sc["contract"]} != {CONTRACT}{STALE}')
assert sc['n_numeric'] == len(sc['numeric_features']), 'n_numeric 불일치'
assert sorted(sc['files']) == ['ftt_2024_tr.parquet', 'ftt_2024_va.parquet'], \
    f'schema.files {sorted(sc["files"])}'

# ---- 4. 압축 해제 후 parquet 이 schema 와 같은 파일인지
DATA.mkdir(parents=True, exist_ok=True)
with tarfile.open(ARCHIVE) as t:
    t.extractall(FTT)
print(f'\nextracted -> {DATA}')
for k, v in sc['files'].items():
    f = DATA / k
    assert f.exists(), f'압축 해제 후 없음: {f}'
    assert f.stat().st_size == v['bytes'], (
        f'{k} 크기 {f.stat().st_size:,} != schema {v["bytes"]:,}')
    d = hashlib.sha256(f.read_bytes()).hexdigest()
    assert d == v['sha256'], f'{k} sha256 불일치\n  {d}\n  {v["sha256"]}'
    print(f'  OK  {k:22s} {v["bytes"]:>12,} bytes  sha256 일치')

# ---- 5. 내용 요약 — 숫자를 눈으로 확인하고 넘어간다
print(f'\ncontract {sc["contract"]}   fold {sc["fold"]}   '
      f'생성 {sc.get("created", "?")}')
print(f'numeric feature   {sc["n_numeric"]}')
print(f'categorical       {len(sc["categorical_features"])}  '
      f'{sc["categorical_features"]}')
print(f'{"file":24s}{"rows":>11s}  {"seasons":22s}{"y mean":>9s}{"p_v9":>12s}')
for k, v in sc['files'].items():
    cov = v['p_v9_rows'] / v['rows'] * 100
    print(f'  {k:22s}{v["rows"]:>11,}  {str(v["seasons"]):22s}'
          f'{v["y_mean"]:>9.4f}{v["p_v9_rows"]:>10,} ({cov:.0f}%)')

tr, va = sc['files']['ftt_2024_tr.parquet'], sc['files']['ftt_2024_va.parquet']
assert max(tr['seasons']) < 2024, f'train 에 2024 유입: {tr["seasons"]}'
assert va['seasons'] == [2024], f'valid 이 2024 단독이 아님: {va["seasons"]}'
print(f'\nsplit    train {tr["seasons"]} < valid {va["seasons"]}   누수 없음')
print(f'model C  p_v9 있는 {tr["p_v9_rows"]:,} 행만 학습 '
      f'({tr["p_v9_rows"]/tr["rows"]*100:.0f}%) — 설계상 정상')


## 6. Model 설정 확인

| | 입력 | 출력 | 학습 행 |
|---|---|---|---|
| **A** | numeric 101 | `p = sigmoid(f(x))` | 1,221,585 |
| **B** | numeric 101 + 범주형 4 임베딩 | `p = sigmoid(f(x))` | 1,221,585 |
| **C** | B 와 같은 몸통 | `p = sigmoid(logit(p_v9) + f(x))` | **492,997 (40%)** |

**C 의 행이 적은 것은 버그가 아닙니다.** C 는 `p_v9` 가 있는 행만 쓸 수 있는데
v9 OOF 가 2022–2024 에만 존재합니다. A/B 와 비교할 때 반드시 감안하십시오.

**세 모델 모두 타깃은 `control_success` 이며 residual target 학습이 아닙니다.**
17차가 residual 경로로 2024 dBSS **−177.0 / −218.0**, corr 0.94~0.996 을 냈습니다.

범주형 4개 중 3개(`pitcher_team_id`, `batter_team_id`, `base_state_c`)는 이미
numeric 101 안에 있습니다 — B 가 얻는 것은 새 컬럼이 아니라 **임베딩 처리**입니다.


In [ ]:
!python experiments/deep_learning/ft_transformer/train_ftt.py --help


## 7. 학습 (seed 42)

Model A → B → C 순서로 학습하고 각 모델의 OOF 예측을 저장합니다.
`patience 3` 으로 조기 종료하므로 15 epoch 을 다 쓰지 않을 수 있습니다.

**예상 40~90분.** 셀이 도는 동안 epoch 별 loss 와 BSS/Res/Rel 이 출력됩니다.


In [ ]:
!python experiments/deep_learning/ft_transformer/train_ftt.py \
    --seed 42 --models A,B,C \
    --data-dir experiments/deep_learning/ft_transformer/data \
    --out-dir experiments/deep_learning/ft_transformer/oof


## 8. Gate 평가

v9 구성요소(`p_A7` / `p_A9` / `p_Bcat`)는 준비된 `ftt_2024_va.parquet` 안에 있으므로
**`work/` 에 접근하지 않습니다.**


In [ ]:
!python experiments/deep_learning/ft_transformer/v0_gate.py \
    --seed 42 --models A,B,C \
    --data-dir experiments/deep_learning/ft_transformer/data \
    --oof-dir experiments/deep_learning/ft_transformer/oof


## 9. 게이트 판정 기준

```
PASS  (모두 만족)
  1. BSS          >= 750
  2. corr(v9)     <  0.85
  3. blend 개선    >= +30
  4. frontier gap >= +57      <-- 1·2 가 독립이 아니므로 함께 봄

FAIL  (하나라도)
  corr >= 0.95   또는   blend 개선 없음
```

**4번이 판정의 핵심입니다.** BSS 와 corr 은 `BSS = 2052·corr − 1051` (R² 0.998) 로
묶여 있어 따로 보면 "왜 떨어졌는지" 를 놓칩니다. `corr = 0.85` 에서 직선이
예측하는 BSS 가 693.2 이므로, 750 을 넘으려면 직선에서 **+57 이상 벗어나야** 합니다.

`blend 개선` 은 4요소 stacking 의 2024 **in-sample 상한** 이 3요소 상한(+19.4)
대비 얼마나 올라갔는가입니다. **in-sample 이므로 배포 가능성 주장이 아닙니다** —
"여지가 있는가" 만 묻습니다.

아래 셀이 실제 수치로 판정을 출력합니다.


In [ ]:
import pandas as pd, pathlib

FTT = pathlib.Path('/content/kbo/experiments/deep_learning/ft_transformer')
res = pd.read_csv(FTT / 'v0_results_s42.csv')

TH = dict(bss=750.0, corr=0.85, blend=30.0, gap=57.0)
print(f'{"model":>6} {"BSS":>8} {"corr(v9)":>9} {"gap":>8} {"blend":>8}   판정')
print('-' * 60)
verdicts = {}
for _, r in res.iterrows():
    c = dict(BSS=r.BSS >= TH['bss'], corr=r.corr_v9 < TH['corr'],
             blend=r.ceil_gain >= TH['blend'], gap=r.off_frontier >= TH['gap'])
    ok = all(c.values())
    verdicts[r.model] = dict(pass_all=bool(ok), **{k: bool(v) for k, v in c.items()})
    fail = [k for k, v in c.items() if not v]
    print(f'{r.model:>6} {r.BSS:>8.1f} {r.corr_v9:>9.4f} {r.off_frontier:>+8.1f} '
          f'{r.ceil_gain:>+8.1f}   {"PASS" if ok else "FAIL (" + ",".join(fail) + ")"}')

overall = any(v['pass_all'] for v in verdicts.values())
print('\n' + '=' * 60)
print(f'V0 종합: {"PASS" if overall else "FAIL"}')
print('통과 -> seed 43, 44 로 확장' if overall
      else 'FAIL -> seed 확장 없음. 추가 실험 없음. v9 유지.')
print('=' * 60)


## 10. 결과 저장

`results/metrics.json` 과 `results/gate_report.md` 를 만듭니다.
모델 파일·제출물은 만들지 않습니다.


In [ ]:
import json, pathlib, datetime, pandas as pd

FTT = pathlib.Path('/content/kbo/experiments/deep_learning/ft_transformer')
OUTD = FTT / 'results'
OUTD.mkdir(parents=True, exist_ok=True)

res = pd.read_csv(FTT / 'v0_results_s42.csv')
sc = json.loads((FTT / 'data' / 'schema.json').read_text())

metrics = dict(
    experiment='FT-Transformer V0', fold=2024, seed=42,
    created=datetime.datetime.now().isoformat(timespec='seconds'),
    thresholds=TH, frontier=dict(slope=2052.0, intercept=-1051.0, r2=0.998),
    schema=dict(n_numeric=sc['n_numeric'],
                n_categorical=len(sc['categorical_features']),
                files={k: v['rows'] for k, v in sc['files'].items()}),
    models=res.to_dict(orient='records'), verdicts=verdicts,
    overall_pass=bool(overall))
(OUTD / 'metrics.json').write_text(json.dumps(metrics, indent=1, default=float))

L = ['# FT-Transformer V0 — gate report', '',
     f'- fold 2024 · seed 42 · {metrics["created"]}',
     f'- frontier `BSS = 2052·corr − 1051` (R² 0.998)',
     f'- 판정: **{"PASS" if overall else "FAIL"}**', '',
     '| model | BSS | Resolution | Reliability | calib bias | corr(v9) | '
     'frontier gap | blend | 판정 |', '|---|---|---|---|---|---|---|---|---|']
for _, r in res.iterrows():
    v = verdicts[r.model]
    fail = [k for k, x in v.items() if k != 'pass_all' and not x]
    L.append(f'| {r.model} | {r.BSS:.1f} | {r.Resolution:.1f} | '
             f'{r.Reliability:.1f} | {r.calib_bias:+.5f} | {r.corr_v9:.4f} | '
             f'{r.off_frontier:+.1f} | {r.ceil_gain:+.1f} | '
             f'{"PASS" if v["pass_all"] else "FAIL (" + ",".join(fail) + ")"} |')
L += ['', '## 기준', '```',
      'PASS: BSS>=750 AND corr<0.85 AND blend>=+30 AND frontier gap>=+57',
      'FAIL: corr>=0.95 OR blend 개선 없음', '```', '',
      '## 유의', '',
      '- model C 는 p_v9 가 있는 행만 학습하므로 492,997행(40%). A/B 는 1,221,585행.',
      '- blend 는 2024 in-sample stacking 상한이며 배포 가능성 주장이 아님.',
      '- 세 모델 모두 타깃은 control_success. residual target 아님.',
      '- work/submit_v9.zip 무변경, test.csv 미사용, work/ 미사용.']
(OUTD / 'gate_report.md').write_text('\n'.join(L))

BANNED = ('.pkl', '.zip', '.cbm', 'submit', 'work')
made = sorted(p.name for p in OUTD.iterdir())
bad = [n for n in made if any(b in n for b in BANNED)]
assert not bad, f'금지 산출물: {bad}'
print(f'results/ -> {made}   (금지 산출물 0개)')
print(f'\n  Drive 로 복사:')
print(f'  !cp -r {OUTD} /content/drive/MyDrive/kbo/ftt_results')
